# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [1]:
import scipy.io
import numpy as np

# ==================== 1️⃣ 读取 .mat 文件 ====================
mat_data = scipy.io.loadmat(
    '/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_4.mat'
)

data = mat_data['Unit03_1_5_select_4']

print("Original data shape:", data.shape)  # (N, 14)

# ==================== 2️⃣ 设置抽样参数 ====================
n = 10000        # ⭐你只需要改这里
seed = 66      # 可选：保证可复现

np.random.seed(seed)

num_samples = data.shape[0]
assert n <= num_samples, "n cannot be larger than total samples!"

# ==================== 3️⃣ 随机抽取 n 个样本（行） ====================
indices = np.random.choice(num_samples, size=n, replace=False)
data = data[indices, :]

print("Sampled data shape:", data.shape)

# ==================== 4️⃣ 查看抽样结果 ====================
print(data[:5])   # 前 5 行看看


Original data shape: (120000, 14)
Sampled data shape: (10000, 14)
[[ 6.29527760e+00  1.02092609e+01  2.03887776e-01 -1.44377480e+01
   2.61691406e+02  3.05918549e+02  5.76434265e+02  8.14589294e+02
   7.93603638e+02  9.01494019e+02  7.46951904e+02  7.47741699e+02
   7.46326660e+02  9.33002472e+00]
 [ 6.16079855e+00  1.03493671e+01  2.08327323e-01 -1.13825216e+01
   2.61934235e+02  3.14191925e+02  5.66003113e+02  8.36050720e+02
   8.15244690e+02  9.06671814e+02  7.52554993e+02  7.51828125e+02
   7.51284668e+02  5.44592047e+00]
 [ 6.09339237e+00  1.00023880e+01  2.05203310e-01 -1.62196388e+01
   1.21227821e+02  3.07980621e+02  5.98566528e+02  8.26992981e+02
   8.07167542e+02  8.83469177e+02  7.53504639e+02  7.54318115e+02
   7.52330566e+02  1.02919979e+01]
 [ 6.32518101e+00  1.01809092e+01  2.06452131e-01 -1.31864719e+01
   2.62192352e+02  3.15252472e+02  5.65189331e+02  8.31019043e+02
   8.06902588e+02  9.02616333e+02  7.50667725e+02  7.51341370e+02
   7.49867371e+02  6.27227306e+00]
 [

## 特征和标签的分离

In [2]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (10000, 13)
Shape of Labels (y): (10000, 1)


## 三集划分

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (6999, 13) (6999, 1)
Shape of Validation Set (X_val, y_val): (1500, 13) (1500, 1)
Shape of Test Set (X_test, y_test): (1501, 13) (1501, 1)


## 归一化

In [4]:
import numpy as np

# ======================= 1️⃣ X：用 train 的均值和方差 =======================
mu = X_train.mean(axis=0, keepdims=True)     # (1, 13)
std = X_train.std(axis=0, keepdims=True)    # (1, 13)
std[std == 0] = 1e-8

X_train = (X_train - mu) / std
X_val   = (X_val   - mu) / std
X_test  = (X_test  - mu) / std

print("X normalized shapes:")
print(X_train.shape, X_val.shape, X_test.shape)


# ======================= 2️⃣ y：同样只用 train =======================
y_mu = y_train.mean(axis=0, keepdims=True)     # (1,1)
y_std = y_train.std(axis=0, keepdims=True)    # (1,1)
y_std[y_std == 0] = 1e-8

#===========================保留归一化前的y值==============================#
y_train_raw = y_train
y_val_raw = y_val
y_test_raw = y_test


y_train = (y_train - y_mu) / y_std
y_val   = (y_val   - y_mu) / y_std
y_test  = (y_test  - y_mu) / y_std

print("y normalized shapes:")
print(y_train.shape, y_val.shape, y_test.shape)


# ======================= 3️⃣ 反归一化函数 =======================
def inverse_y(y_norm, y_mu, y_std):
    """
    y_norm: normalized prediction, shape [N,1] or [N]
    y_mu, y_std: from training set
    """
    return y_norm * y_std + y_mu


X normalized shapes:
(6999, 13) (1500, 13) (1501, 13)
y normalized shapes:
(6999, 1) (1500, 1) (1501, 1)


## 样本滑窗

In [5]:
import torch
import numpy as np

def sliding_window(
    X,
    y,
    window_size,
    stride=1
):
    """
    X: [N, D]
    y: [N] or [N, 1]
    window_size: T
    stride: step between windows
    """

    if isinstance(X, np.ndarray):
        X = torch.from_numpy(X).float()
    if isinstance(y, np.ndarray):
        y = torch.from_numpy(y).float()

    assert X.dim() == 2
    assert len(X) == len(y)

    X_seq, y_seq = [], []

    for end in range(window_size - 1, len(X), stride):
        start = end - window_size + 1
        X_seq.append(X[start:end + 1])
        y_seq.append(y[end])

    return (
        torch.stack(X_seq),           # [N', T, D]
        torch.stack(y_seq).view(-1, 1)
    )


T = 20  # 滑窗长度，你之后可以调

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float()#每一个滑窗样本的标签 y，取的是“窗口最后一个时间点”的真实值

X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).float()

X_test  = torch.from_numpy(X_test).float()
y_test  = torch.from_numpy(y_test).float()


X_train_seq, y_train_seq = sliding_window(
    X_train, y_train, window_size=T
)
X_val_seq, y_val_seq = sliding_window(
    X_val, y_val, window_size=T
)
X_test_seq, y_test_seq = sliding_window(
    X_test, y_test, window_size=T
)
print(X_train_seq.shape)  # [N', T, D]
print(y_train_seq.shape)  # [N', 1]


torch.Size([6980, 20, 13])
torch.Size([6980, 1])


# 模型

## RNN模型定义

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KANLinear(nn.Module):
    """
    KAN-style linear layer:
    y = sum_i f_i(x_i), where f_i is a learnable 1D function
    implemented via basis expansion.
    """
    def __init__(self, in_features, num_basis=8):
        super().__init__()
        self.in_features = in_features
        self.num_basis = num_basis

        # 每个维度一组 basis 权重
        self.weight = nn.Parameter(
            torch.randn(in_features, num_basis) * 0.1
        )
        self.bias = nn.Parameter(torch.zeros(1))

        # 固定 basis centers（[-1, 1]）
        centers = torch.linspace(-1, 1, num_basis)
        self.register_buffer("centers", centers)

        self.gamma = nn.Parameter(torch.ones(1))  # 控制平滑度

    def forward(self, x):
        """
        x: [B, in_features]
        """
        # x -> [B, in_features, num_basis]
        x_exp = x.unsqueeze(-1)

        # RBF basis
        basis = torch.exp(
            -self.gamma * (x_exp - self.centers) ** 2
        )

        # 加权求和
        y = (basis * self.weight).sum(dim=(1, 2)) + self.bias
        return y.unsqueeze(-1)


class FeatureGATFusion(nn.Module):
    """
    Feature-wise GAT fusion:
    x [B,T,D] -> GAT over features -> [B,T,embed_dim]
    """
    def __init__(
        self,
        num_features,      # D
        embed_dim=32,
        dropout=0.1,
        alpha=0.2
    ):
        super().__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim

        # 标量特征 -> 节点 embedding
        self.W = nn.Linear(1, embed_dim, bias=False)

        # 注意力参数
        self.attn = nn.Linear(2 * embed_dim, 1, bias=False)

        self.leaky_relu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: [B,T,D]
        return: [B,T,embed_dim]
        """
        B, T, D = x.shape
        assert D == self.num_features

        # [B,T,D] -> [B*T, D, 1]
        xt = x.reshape(B * T, D).unsqueeze(-1)

        # 节点嵌入
        h = self.W(xt)                         # [B*T, D, embed_dim]

        # 构造 (i,j) 特征对
        h_i = h.unsqueeze(2).expand(-1, D, D, -1)
        h_j = h.unsqueeze(1).expand(-1, D, D, -1)

        # 注意力分数
        e = self.leaky_relu(
            self.attn(torch.cat([h_i, h_j], dim=-1))
        ).squeeze(-1)                          # [B*T, D, D]

        # softmax：每个特征对“其余特征”的权重
        alpha = F.softmax(e, dim=-1)
        alpha = self.dropout(alpha)

        # 特征融合
        h_new = torch.matmul(alpha, h)         # [B*T, D, embed_dim]

        # 聚合所有特征节点
        h_fused = h_new.mean(dim=1)             # [B*T, embed_dim]

        # reshape 回时间维
        h_fused = h_fused.view(B, T, self.embed_dim)

        return h_fused

class RNNRegressor(nn.Module):
    def __init__(
        self,
        input_dim,              # 原始特征数 D
        hidden_dim=64,
        num_layers=1,
        rnn_type="LSTM",
        dropout=0.0,

        # ===== Feature-GAT =====
        gat_embed_dim=32,

        # ===== KAN =====
        kan_basis=8
    ):
        super().__init__()

        # ============ 1️⃣ 特征维 GAT ============
        self.feature_gat = FeatureGATFusion(
            num_features=input_dim,
            embed_dim=gat_embed_dim,
            dropout=dropout
        )

        rnn_input_dim = gat_embed_dim
        self.rnn_type = rnn_type.upper()

        # ============ 2️⃣ LSTM / GRU / RNN ============
        if self.rnn_type == "RNN":
            self.rnn = nn.RNN(
                rnn_input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        elif self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                rnn_input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        elif self.rnn_type == "GRU":
            self.rnn = nn.GRU(
                rnn_input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        else:
            raise ValueError("rnn_type must be RNN / LSTM / GRU")

        # ============ 3️⃣ KAN 回归头 ============
        self.regressor = KANLinear(
            in_features=hidden_dim,
            num_basis=kan_basis
        )

    def forward(self, x):
        """
        x: [B, T, D]
        """
        # ① 特征间 GAT 融合
        x = self.feature_gat(x)        # [B, T, gat_embed_dim]

        # ② 时间建模
        out, _ = self.rnn(x)           # [B, T, hidden_dim]
        h_last = out[:, -1, :]         # [B, hidden_dim]

        # ③ 回归
        y_hat = self.regressor(h_last)
        return y_hat


    
def train_one_epoch(model, optimizer, criterion, X, y, batch_size=64):
    model.train()
    total_loss = 0.0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        optimizer.zero_grad()
        y_hat = model(xb).view(-1)
        loss = criterion(y_hat, yb.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(X)


@torch.no_grad()
def evaluate(model, criterion, X, y, batch_size=64):
    model.eval()
    total_loss = 0.0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        y_hat = model(xb).view(-1)
        loss = criterion(y_hat, yb.view(-1))

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(X)

from sklearn.metrics import r2_score

@torch.no_grad()
def evaluate_r2(model, X, y, batch_size=64):
    model.eval()

    y_true_list = []
    y_pred_list = []

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        y_hat = model(xb)

        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)

    return r2_score(y_true, y_pred)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 假设你已经是 torch.Tensor
X_train_seq = X_train_seq.to(device)
y_train_seq = y_train_seq.to(device)
X_val_seq   = X_val_seq.to(device)
y_val_seq   = y_val_seq.to(device)

model = RNNRegressor(
    input_dim=13,        # 特征数
    hidden_dim=64,
    num_layers=2,
    rnn_type="LSTM",
    dropout=0.1,

    gat_embed_dim=13,    # ⭐ GAT 的输出维度

    kan_basis=8
).to(device)

model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 50

best_val_r2 = -float("inf")   # R² 越大越好
best_epoch = -1

for epoch in range(1, num_epochs + 1):

    # ===== 1️⃣ 训练 =====
    train_loss = train_one_epoch(
        model, optimizer, criterion,
        X_train_seq, y_train_seq
    )

    # ===== 2️⃣ 验证 MSE =====
    val_loss = evaluate(
        model, criterion,
        X_val_seq, y_val_seq
    )

    # ===== 3️⃣ 计算 R² =====
    train_r2 = evaluate_r2(
        model, X_train_seq, y_train_seq
    )
    val_r2 = evaluate_r2(
        model, X_val_seq, y_val_seq
    )

    # ===== 4️⃣ 保存最优模型（按 val R²）=====
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_epoch = epoch
        torch.save(model.state_dict(), "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression_v2/result/model_save/best_RNN_model.pt")

    # ===== 5️⃣ 打印日志 =====
    print(
        f"[Epoch {epoch:03d}] "
        f"Train MSE: {train_loss:.4f} | "
        f"Val MSE: {val_loss:.4f} | "
        f"Train R²: {train_r2:.4f} | "
        f"Val R²: {val_r2:.4f}"
    )

print(
    f"\n✅ Best model saved at epoch {best_epoch}, "
    f"Val R² = {best_val_r2:.4f}"
)



[Epoch 001] Train MSE: 0.9896 | Val MSE: 0.9985 | Train R²: 0.0427 | Val R²: 0.0286
[Epoch 002] Train MSE: 0.8008 | Val MSE: 0.8650 | Train R²: 0.2165 | Val R²: 0.1585
[Epoch 003] Train MSE: 0.7627 | Val MSE: 0.8307 | Train R²: 0.2409 | Val R²: 0.1919
[Epoch 004] Train MSE: 0.7585 | Val MSE: 0.8200 | Train R²: 0.2500 | Val R²: 0.2023
[Epoch 005] Train MSE: 0.7512 | Val MSE: 0.8195 | Train R²: 0.2500 | Val R²: 0.2028
[Epoch 006] Train MSE: 0.7499 | Val MSE: 0.8097 | Train R²: 0.2596 | Val R²: 0.2123
[Epoch 007] Train MSE: 0.7456 | Val MSE: 0.8093 | Train R²: 0.2607 | Val R²: 0.2128
[Epoch 008] Train MSE: 0.7492 | Val MSE: 0.8070 | Train R²: 0.2630 | Val R²: 0.2150
[Epoch 009] Train MSE: 0.7455 | Val MSE: 0.8064 | Train R²: 0.2647 | Val R²: 0.2155
[Epoch 010] Train MSE: 0.7420 | Val MSE: 0.8088 | Train R²: 0.2632 | Val R²: 0.2132
[Epoch 011] Train MSE: 0.7443 | Val MSE: 0.8057 | Train R²: 0.2656 | Val R²: 0.2162
[Epoch 012] Train MSE: 0.7409 | Val MSE: 0.8069 | Train R²: 0.2656 | Val R²:

# 测试

In [9]:
# ======================= 单 cell：Test 评估（device-safe） =======================
import torch
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

# ---------- 1️⃣ 统一 device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
X_test_seq = X_test_seq.to(device)
y_test_seq = y_test_seq.to(device)

# ---------- 2️⃣ 加载最优模型 ----------
model.load_state_dict(torch.load("/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression_v2/result/model_save/best_RNN_model.pt", map_location=device))
model.eval()

# ---------- 3️⃣ Test 预测 ----------
y_true_list = []
y_pred_list = []

batch_size = 64

with torch.no_grad():
    for i in range(0, len(X_test_seq), batch_size):
        xb = X_test_seq[i:i+batch_size]
        yb = y_test_seq[i:i+batch_size]

        y_hat = model(xb)

        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

y_true = np.concatenate(y_true_list)
y_pred = np.concatenate(y_pred_list)

# ---------- 4️⃣ 计算指标 ----------
test_mse = mean_squared_error(y_true, y_pred)
test_r2  = r2_score(y_true, y_pred)

print("🧪 Test results")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R² : {test_r2:.4f}")


🧪 Test results
Test MSE: 0.7070
Test R² : 0.2548


/tmp/ipykernel_323856/1227843688.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home/charles/HZU/Industrial_Software_Testing/Industr